# 7. Data sources and audits

This notebook consolidates source downloads, salary-source checks, and PCA-feature auditing. Network operations and file rewrites are opt-in. The verification and audit cells are separate so you can inspect results before changing any source-review records.

## 1. Packages and project paths

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from zipfile import ZipFile, is_zipfile
import csv
import hashlib
import io
import json
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'raw').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'raw').exists(): raise FileNotFoundError('Open the notebook inside the project.')
rawPath = rootPath / 'data' / 'raw'
processedPath = rootPath / 'data' / 'processed'


## 2. Source manifest and download validation

In [ ]:
SOURCES = {
    "basketball_reference_latest.zip": {"url": "https://www.kaggle.com/api/v1/datasets/download/sumitrodatta/nba-aba-baa-stats", "members": ["Advanced.csv", "Player Per Game.csv", "Player Season Info.csv"], "directory": "basketball_reference"},
    "salaries/historical_salaries.csv": {"url": "https://raw.githubusercontent.com/logan-lauton/NBA-Player-HUD/main/NBA%20Salaries%281990-2023%29.csv", "required_columns": ["playerName", "seasonStartYear", "salary"]},
    "salaries/bbref_historical_salaries.csv": {"url": "https://raw.githubusercontent.com/ramvibhakar/basketball-analytics/master/data/player_salary.csv", "required_columns": ["season", "url", "salary", "team_url"]},
    "salaries/alternative_salaries.csv": {"url": "https://raw.githubusercontent.com/erikgregorywebb/datasets/master/nba-salaries.csv", "required_columns": ["name", "salary", "season"]},
    "salaries/hoopshype_snapshot.csv": {"url": "https://huggingface.co/datasets/Mr-Bridge/nba-salary-cap-contracts-2016-2026/resolve/main/player_salaries.csv", "required_columns": ["player", "salary", "season_year", "team_id"]}
}

for year in range(1997, 2027):
    filename = "sportsdataverse/player_season_stats_" + str(year) + ".parquet"
    SOURCES[filename] = {"url": "https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_player_season_stats/player_season_stats_" + str(year) + ".parquet"}

for year in range(2010, 2027):
    filename = "bbref_salaries/bbref_salaries_" + str(year) + ".parquet"
    SOURCES[filename] = {"url": "https://basketball-data.billmill.org/nba/bbref/bbref_salaries/season=" + str(year) + "/data.parquet"}

def validateDownload(blob, filename, source):
    members = {}

    if "members" in source:
        if not is_zipfile(io.BytesIO(blob)):
            raise ValueError(filename + ": response is not a ZIP archive")

        with ZipFile(io.BytesIO(blob)) as archive:
            for member in source["members"]:
                if member not in archive.namelist():
                    raise ValueError(filename + ": missing expected file " + member)

                members[member] = archive.read(member)

    elif filename.endswith(".parquet"):
        if blob[:4] != b"PAR1" or blob[-4:] != b"PAR1":
            raise ValueError(filename + ": response is not a Parquet file")

    else:
        reader = csv.DictReader(io.StringIO(blob.decode("utf-8-sig")))

        for column in source["required_columns"]:
            if reader.fieldnames is None or column not in reader.fieldnames:
                raise ValueError(filename + ": missing expected column " + column)

    return members

### Download or verify source files

In [ ]:
def downloadSources(outputPath=rawPath, refresh=False):
    outputPath.mkdir(parents=True, exist_ok=True)
    records = []
    for filename, source in SOURCES.items():
        filePath = outputPath / filename
        filePath.parent.mkdir(parents=True, exist_ok=True)
        if refresh or not filePath.exists():
            print('Downloading ' + filename, flush=True)
            for attempt in range(3):
                try:
                    request = Request(source['url'], headers={'User-Agent':'BargainBall-academic-project/1.0'})
                    with urlopen(request, timeout=120) as response: blob = response.read()
                    members = validateDownload(blob, filename, source)
                    temporaryPath = filePath.with_suffix(filePath.suffix + '.part')
                    temporaryPath.write_bytes(blob)
                    temporaryPath.replace(filePath)
                    break
                except Exception:
                    if attempt == 2: raise
        else:
            blob = filePath.read_bytes()
            members = validateDownload(blob, filename, source)
        for member, memberBytes in members.items():
            memberPath = outputPath / source['directory'] / member
            memberPath.parent.mkdir(parents=True, exist_ok=True)
            memberPath.write_bytes(memberBytes)
        records.append({'file':filename,'url':source['url'],'bytes':len(blob),'sha256':hashlib.sha256(blob).hexdigest(),'verified_utc':datetime.now(timezone.utc).isoformat()})
    curatedLookup = outputPath / 'salaries' / 'BargainBall_listed_salaries.csv'
    if not curatedLookup.exists(): raise FileNotFoundError('Keep the bundled reconciled salary lookup in data/raw/salaries.')
    records.append({'file':str(curatedLookup.relative_to(outputPath)),'source':'previous project salary reconciliation','bytes':curatedLookup.stat().st_size,'sha256':hashlib.sha256(curatedLookup.read_bytes()).hexdigest()})
    manifestPath = outputPath / 'download_manifest.json'
    manifestPath.write_text(json.dumps(records,indent=2) + '\n',encoding='utf-8')
    print('Verified source manifest: ' + str(manifestPath))

runSourceDownload = False
if runSourceDownload: downloadSources(refresh=False)
print('Source download enabled: ' + str(runSourceDownload))

## 3. Name matching and salary cross-check helpers

In [ ]:
name_aliases = {
    "Isaac Austin": "Ike Austin",
    "Adama-Alpha Bal": "Adama Bal",
    "Walter Clayton": "Walter Clayton Jr.",
    "Egor Dёmin": "Egor Dëmin",
    "Isaac Fontaine": "Ike Fontaine",
    "Kiwane Lemorris Garris": "Kiwane Garris",
    "DaRon Holmes": "DaRon Holmes II",
    "Eugene Jeter": "Pooh Jeter",
    "Ibo Kutluay": "Ibrahim Kutluay",
    "Rich Manning": "Richard Manning",
    "Stanislav Medvedenko": "Slava Medvedenko",
    "Nathan Mensah": "Nate Mensah",
    "Ronald Murray": "Flip Murray",
    "Norm Richardson": "Norman Richardson",
    "Danny Schayes": "Dan Schayes",
    "Ha Seung-Jin": "Ha Ha",
    "Mike Sweetney": "Michael Sweetney",
    "Clarence Weatherspoon": "Clar. Weatherspoon",
    "Sun Yue": "Sun Sun",
    "Cui Yongxi": "Cui Cui",
    "Wade Baldwin": "Wade Baldwin IV",
    "Brandon Boston Jr.": "Brandon Boston",
    "Brian Bowen": "Brian Bowen II",
    "Reggie Bullock": "Reggie Bullock Jr.",
    "Jimmy Butler": "Jimmy Butler III",
    "John Butler": "John Butler Jr.",
    "Mitch Creek": "Mitchell Creek",
    "Jeff Dowtin": "Jeff Dowtin Jr.",
    "Vince Edwards": "Vincent Edwards",
    "Vítor Luiz Faverani": "Vitor Faverani",
    "Melvin Frazier": "Melvin Frazier Jr.",
    "Michael Frazier": "Michael Frazier II",
    "Harry Giles": "Harry Giles III",
    "Nigel Hayes": "Nigel Hayes-Davis",
    "Ron Holland": "Ronald Holland II",
    "Vince Hunter": "Vincent Hunter",
    "Matthew Hurt": "Matt Hurt",
    "GG Jackson II": "GG Jackson",
    "Trey Jemison": "Trey Jemison III",
    "Perry Jones": "Perry Jones III",
    "Kevin Knox": "Kevin Knox II",
    "Roger Mason": "Roger Mason Jr.",
    "Marcus Morris": "Marcus Morris Sr.",
    "Zach Norvell": "Zach Norvell Jr.",
    "Johnny O'Bryant": "Johnny O'Bryant III",
    "Bobby Portis": "Bobby Portis Jr.",
    "Cameron Reynolds": "Cam Reynolds",
    "Glen Rice Jr.": "Glen Rice",
    "Jermaine Samuels": "Jermaine Samuels Jr.",
    "Tre Scott": "Trevon Scott",
    "Jeff Taylor": "Jeffery Taylor",
    "Xavier Tillman Sr.": "Xavier Tillman",
    "Derrick Walton": "Derrick Walton Jr.",
    "Andrew White": "Andrew White III",
    "Jeenathan Williams": "Nate Williams",
    "Matt Williams": "Matt Williams Jr.",
    "Robert Williams": "Robert Williams III"
}

explicit_ids = {
    "mitchto02": "203502",
    "mitchto03": "203183",
    "johnsch03": "202419",
    "johnsch04": "203187",
    "willima03": "200766",
    "willima04": "201173",
    "jonesch01": "279",
    "jonesch03": "1869",
    "smithst01": "120"
}

def normalize_name(name):
    name = name.casefold()
    name = name.replace("ı", "i")
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")

    cleaned_name = ""

    for character in name:
        if character.isalnum():
            cleaned_name = cleaned_name + character

    return cleaned_name

def name_key(name):
    name = name_aliases.get(name, name)
    return normalize_name(name)

### Cross-check flagged salary records

In [ ]:
BBREF_URL = 'https://github.com/ramvibhakar/basketball-analytics/blob/master/data/player_salary.csv'
ALT_URL = 'https://github.com/erikgregorywebb/datasets/blob/master/nba-salaries.csv'
HIST_URL = 'https://github.com/logan-lauton/NBA-Player-HUD/blob/main/NBA%20Salaries(1990-2023).csv'
salaryRawPath = rawPath / 'salaries'
def dollars(series):
    return pd.to_numeric(
        series.astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False),
        errors="coerce",
    )

def runSalarySourceAudit():
    pairs = pd.read_csv(processedPath / 'salary_cap_normalized_pairs.csv')
    flagged = pairs[pairs["salary_available_check"] & pairs["salary_needs_source_review"]].copy()
    flagged["name_key"] = flagged["player"].map(name_key)

    bbref = pd.read_csv(salaryRawPath / 'bbref_historical_salaries.csv')
    bbref["player_id"] = bbref["url"].str.extract(r"/players/[a-z]/([^/.]+)")
    bbref["target_season"] = bbref["season"].str[:4].astype(int) + 1
    bbref["bbref_amount"] = dollars(bbref["salary"])
    bbref = bbref.groupby(["player_id", "target_season"], as_index=False)["bbref_amount"].sum()

    alt = pd.read_csv(salaryRawPath / 'alternative_salaries.csv')
    alt["name_key"] = alt["name"].map(name_key)
    alt["target_season"] = pd.to_numeric(alt["season"], errors="raise").astype(int)
    alt["alt_amount"] = pd.to_numeric(alt["salary"], errors="coerce")
    alt = alt.groupby(["name_key", "target_season"], as_index=False)["alt_amount"].sum()

    # Resolve names through the project crosswalk, and reject source name-season
    # collisions instead of summing salaries that may belong to different players.
    names = pairs[["player_id", "target_season", "name_key"]].drop_duplicates()
    names = names[~names.duplicated(["name_key", "target_season"], keep=False)]
    historical = pd.read_csv(salaryRawPath / 'historical_salaries.csv').drop_duplicates()
    historical["name_key"] = historical["playerName"].map(name_key)
    historical["target_season"] = historical["seasonStartYear"].astype(int) + 1
    historical["hist_amount"] = dollars(historical["salary"])
    historical = historical.merge(names, on=["name_key", "target_season"], how="inner")
    historical = historical.groupby(["player_id", "target_season"], as_index=False).agg(
        hist_amount=("hist_amount", "first"),
        hist_source_rows=("hist_amount", "size"),
    )
    historical.loc[historical["hist_source_rows"].ne(1), "hist_amount"] = pd.NA

    audit = flagged.merge(bbref, on=["player_id", "target_season"], how="left")
    audit = audit.merge(alt, on=["name_key", "target_season"], how="left")
    audit = audit.merge(historical, on=["player_id", "target_season"], how="left")

    # Don't count the originating dataset as corroboration.
    audit["bbref_exact"] = audit["bbref_amount"].notna() & (audit["target_salary"] - audit["bbref_amount"]).abs().le(1)
    audit["alt_exact"] = audit["alt_amount"].notna() & (audit["target_salary"] - audit["alt_amount"]).abs().le(1)
    audit["hist_exact"] = audit["hist_amount"].notna() & (audit["target_salary"] - audit["hist_amount"]).abs().le(1)
    audit["independent_exact_sources"] = ""
    audit["verification_source_urls"] = ""

    for idx, row in audit.iterrows():
        exact_sources = []
        urls = []
        method = row["salary_method"]
        if method not in ("historical_bbref_team_sum",) and row["bbref_exact"]:
            exact_sources.append("Basketball-Reference-ID archive")
            urls.append(BBREF_URL)
        if method != "alternative_listed_salary" and row["alt_exact"]:
            exact_sources.append("alternative salary archive")
            urls.append(ALT_URL)
        if method != "historical_listed_salary" and row["hist_exact"]:
            exact_sources.append("historical listed-salary archive")
            urls.append(HIST_URL)
        audit.at[idx, "independent_exact_sources"] = "; ".join(exact_sources)
        audit.at[idx, "verification_source_urls"] = "; ".join(urls)

    audit["review_outcome"] = "unmatched"
    audit.loc[audit["bbref_amount"].notna() | audit["alt_amount"].notna() | audit["hist_amount"].notna(), "review_outcome"] = "source_difference_or_non-independent_match"
    audit.loc[audit["independent_exact_sources"].ne(""), "review_outcome"] = "verified_exact_crossmatch"
    columns = [
        "player", "player_id", "target_season", "target_salary", "salary_method",
        "salary_source_url", "bbref_amount", "bbref_exact", "alt_amount", "alt_exact",
        "hist_amount", "hist_exact", "independent_exact_sources",
        "verification_source_urls", "review_outcome",
    ]
    audit[columns].to_csv(processedPath / 'salary_source_review_audit.csv', index=False)

    verified = audit[audit["review_outcome"].eq("verified_exact_crossmatch")].copy()
    overrides = verified[["player_id", "target_season", "target_salary", "verification_source_urls"]].copy()
    overrides = overrides.rename(columns={
        "verification_source_urls": "salary_source_url",
    })
    overrides["salary_needs_source_review"] = False
    overrides["salary_method"] = "verified_external_crossmatch"
    overrides_path = salaryRawPath / 'verified_salary_overrides.csv'
    if overrides_path.exists():
        previous = pd.read_csv(overrides_path)
        overrides = pd.concat([previous, overrides], ignore_index=True)
        overrides = overrides.drop_duplicates(["player_id", "target_season"], keep="last")
    overrides.to_csv(overrides_path, index=False)

    print("Flagged rows audited: " + str(len(audit)))
    print("Exact independent crossmatches: " + str(len(verified)))
    print("Still unresolved/conflicting: " + str(len(audit) - len(verified)))
    print(audit["review_outcome"].value_counts().to_string())
    print("Audit: " + str(processedPath / 'salary_source_review_audit.csv'))

In [ ]:
runSalaryArchiveAudit = False
if runSalaryArchiveAudit: runSalarySourceAudit()
print('Salary archive cross-check enabled: ' + str(runSalaryArchiveAudit))

## 4. PCA feature-definition audit

Compare alternative PCA feature definitions on actual training targets only. This audit does not refit or change the active profile model.

In [ ]:
ROOT = rootPath
BOOTSTRAPS = 20

SEED = 42

SHOOTING_COLUMNS = ["two_point_makes", "two_point_misses", "three_point_makes", "three_point_misses", "free_throw_makes", "free_throw_misses"]

BOX_COLUMNS = SHOOTING_COLUMNS + ["offensive_rebounds", "defensive_rebounds", "assists", "steals", "blocks", "turnovers"]

FEATURE_SETS = {
    "advanced_3": ["pie", "ws_48", "bpm"],
    "basic_6": ["points", "rebounds", "assists", "steals", "blocks", "turnovers"],
    "raw_12": BOX_COLUMNS,
    "raw_14_with_minutes": BOX_COLUMNS + ["personal_fouls", "minutes_per_game"],
    "raw_15_with_availability": BOX_COLUMNS + ["personal_fouls", "minutes_per_game", "games"],
    "raw_14_plus_advanced_17": BOX_COLUMNS + ["personal_fouls", "minutes_per_game", "pie", "ws_48", "bpm"],
}

RAW_COLUMNS = {
    "two_point_makes": "x2p_per_game",
    "three_point_makes": "x3p_per_game",
    "free_throw_makes": "ft_per_game",
    "offensive_rebounds": "orb_per_game",
    "defensive_rebounds": "drb_per_game",
    "assists": "ast_per_game",
    "steals": "stl_per_game",
    "blocks": "blk_per_game",
    "turnovers": "tov_per_game",
    "personal_fouls": "pf_per_game",
    "minutes_per_game": "mp_per_game",
    "games": "g",
    "points": "pts_per_game",
    "rebounds": "trb_per_game",
}

def keep_combined_seasons(dataframe):
    kept_rows = []

    for key, group in dataframe.groupby(["player_id", "season"]):
        if len(group) == 1:
            kept_rows.append(group)
        else:
            combined = group[group["team"].str.fullmatch(r"TOT|[2-9][0-9]*TM")]

            if len(combined) != 1:
                raise ValueError("Expected one combined record for " + str(key))

            kept_rows.append(combined)

    return pd.concat(kept_rows, ignore_index=True)

def runPcaFeatureAuditWorkflow():
    processed = ROOT / "data" / "processed"
    rows = pd.read_csv(processed / "train_rows_rates.csv")
    source_path = ROOT / "data" / "raw" / "basketball_reference" / "Player Per Game.csv"
    source = pd.read_csv(source_path)
    source = source[(source["lg"] == "NBA") & source["season"].isin(rows["target_season"])].copy()
    source = keep_combined_seasons(source)
    source = source.rename(columns={"season": "target_season"})
    next_rows = rows[["player_id", "target_season"]].merge(source, on=["player_id", "target_season"], how="left", sort=False, validate="one_to_one", indicator=True)

    if(not next_rows["_merge"].eq("both").all()):
        raise ValueError("Every training target must match an actual next-season record.")

    frame = pd.DataFrame(index=rows.index)

    for name, column in RAW_COLUMNS.items():
        frame[name] = next_rows[column].astype(float)

    frame["two_point_misses"] = next_rows["x2pa_per_game"] - next_rows["x2p_per_game"]
    frame["three_point_misses"] = next_rows["x3pa_per_game"] - next_rows["x3p_per_game"]
    frame["free_throw_misses"] = next_rows["fta_per_game"] - next_rows["ft_per_game"]

    with np.load(processed / "model_data_rates.npz", allow_pickle=False) as data:
        mask = data["rate_mask_train"].astype(bool)

        for target in ["pie", "ws_48", "bpm"]:
            frame[target] = data["y_" + target + "_train"].astype(np.float64)

    # Same training cohort for every candidate; targets are never imputed.
    finite = np.isfinite(frame.to_numpy()).all(axis=1)
    valid = mask & finite
    retained = frame.loc[valid].reset_index(drop=True)

    for name in ["two_point_misses", "three_point_misses", "free_throw_misses"]:
        if((retained[name] < -1e-10).any()):
            raise ValueError("Shot makes cannot exceed attempts.")

    output = ROOT / "results" / "training" / "pca_feature_audit" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output.mkdir(parents=True, exist_ok=False)
    retained.corr().to_csv(output / "feature_correlations.csv")
    summaries = []
    loadings = []

    for name, columns in FEATURE_SETS.items():
        values = retained[columns].to_numpy(dtype=np.float64)
        scaler = StandardScaler()
        standardized = scaler.fit_transform(values)
        pca = PCA(svd_solver="full")
        pca.fit(standardized)
        weights = pca.components_[0].copy()

        if(weights.sum() < 0):
            weights = -weights

        cumulative = np.cumsum(pca.explained_variance_ratio_)
        correlations = retained[columns].corr().to_numpy()
        strong_pairs = 0

        for i in range(len(columns)):
            for j in range(i + 1, len(columns)):
                if(abs(correlations[i, j]) >= 0.9):
                    strong_pairs = strong_pairs + 1

        similarities = []
        rng = np.random.default_rng(SEED)

        for repeat in range(BOOTSTRAPS):
            sampled = values[rng.integers(0, len(values), len(values))]
            sampled_scaled = StandardScaler().fit_transform(sampled)
            fitted = PCA(n_components=1, svd_solver="full").fit(sampled_scaled)
            similarities.append(abs(float(np.dot(weights, fitted.components_[0]))))

        summaries.append({"feature_set": name, "features": len(columns), "training_examples": len(values), "pc1_variance_percent": float(100.0 * pca.explained_variance_ratio_[0]), "components_for_80_percent": int(np.searchsorted(cumulative, 0.8) + 1), "components_for_90_percent": int(np.searchsorted(cumulative, 0.9) + 1), "correlation_pairs_at_least_0_9": strong_pairs, "bootstrap_pc1_cosine_median": float(np.median(similarities)), "bootstrap_pc1_cosine_minimum": float(np.min(similarities))})

        for i, column in enumerate(columns):
            loadings.append({"feature_set": name, "feature": column, "pc1_weight": float(weights[i]), "mean": float(scaler.mean_[i]), "std": float(scaler.scale_[i])})

    summary = pd.DataFrame(summaries)
    summary.to_csv(output / "candidate_summary.csv", index=False)
    pd.DataFrame(loadings).to_csv(output / "pc1_loadings.csv", index=False)
    definitions = {"feature_sets": FEATURE_SETS, "raw_columns": RAW_COLUMNS, "derived_misses": "Attempts minus makes, per game", "cohort": "Original valid-rate training rows with finite raw targets; same rows for every candidate", "original_training_rows": len(rows), "retained_training_rows": int(valid.sum()), "excluded_rate_or_nonfinite_rows": int((~valid).sum()), "raw_source_sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(), "bootstrap_repeats": BOOTSTRAPS, "seed": SEED, "bootstrap_interpretation": "Exploratory row-resampling stability, not player-clustered uncertainty or predictive validation", "salary_in_pca": False, "age_position_in_pca": False, "dev_evaluated": False, "test_evaluated": False}
    (output / "feature_definitions.json").write_text(json.dumps(definitions, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print (summary.to_string(index=False))
    print ("\n14-feature candidate loadings:")
    print (pd.DataFrame(loadings).query("feature_set == 'raw_14_with_minutes'")[["feature", "pc1_weight"]].to_string(index=False))
    print ("\nAudit saved to: " + str(output))
    return output

In [ ]:
runPcaFeatureAudit = False
if runPcaFeatureAudit:
    pcaAuditFolder = runPcaFeatureAuditWorkflow()
    candidateSummary = pd.read_csv(pcaAuditFolder / 'candidate_summary.csv')
    display(candidateSummary)
    pcaFigure, pcaAxis = plt.subplots(figsize=(9,5))
    pcaAxis.bar(candidateSummary.feature_set, candidateSummary.pc1_variance_percent, color='#3a86ff')
    pcaAxis.set_ylabel('Variance explained by PC1 (%)')
    pcaAxis.set_title('Training-only PCA feature audit')
    pcaAxis.tick_params(axis='x',rotation=30)
    pcaFigure.tight_layout()
    display(pcaFigure)
else:
    print('PCA feature audit disabled; set runPcaFeatureAudit = True to run.')

## 5. Online salary checks

The recorded online salary verification ledger and overrides are included as an explicit, opt-in workflow. Run only if you intend to update these local review artifacts.

In [ ]:
ONLINE_SALARY_SOURCES = {
    "MarJon Beauchamp": (
        "https://www.aol.com/articles/2026-nba-free-agent-rankings-143040000.html",
        "AOL/USA Today salary-history listing reports 2025-26 earnings of $347,451.",
    ),
    "Jamaree Bouyea": (
        "https://m.aiscore.com/salary/basketball/player-jamaree-bouyea-o17p8s082vajkjw",
        "AiScore salary history reports 2025-26 annual salary of $878,375.",
    ),
    "Colin Castleton": (
        "https://www.hoopsrumors.com/2025/10/magic-converting-colin-castleton-to-two-way-deal.html",
        "Hoops Rumors reports the 2025-26 two-way salary as $636,435.",
    ),
    "Isaiah Crawford": (
        "https://nbaanalysis.net/houston-rockets-2025-26-payroll-revealed-kevin-durant-joins-team-and-adds-54-7-million-salary-to-the-books/",
        "NBAAnalysis payroll list reports $636,435 for Crawford.",
    ),
    "Trentyn Flowers": (
        "https://m.aiscore.com/salary/basketball/player-trentyn-flowers-zrknxso1oma0qle",
        "AiScore and EuroBasket salary listings report 2025-26 salary of $250,937.",
    ),
    "Enrique Freeman": (
        "https://www.aol.com/articles/2026-nba-free-agent-rankings-143040000.html",
        "AOL/USA Today salary-history listing reports 2025-26 earnings of $636,435.",
    ),
    "Harrison Ingram": (
        "https://www.expressnews.com/sports/spurs/article/2025-26-spurs-get-season-s-roster-breakdown-21097182.php",
        "Express-News Spurs roster breakdown lists 2025-26 salary as $636,435.",
    ),
    "Trey Jemison": (
        "https://www.hoopsrumors.com/2025/12/the-knicks-players-who-benefited-most-from-nba-cup-prize-money.html",
        "Hoops Rumors lists Jemison’s 2025-26 two-way base salary as $636,435.",
    ),
    "Colby Jones": (
        "https://www.spotrac.com/nba/player/_/id/82320/colby-jones",
        "Spotrac salary/cash table reports $85,300 for 2025-26.",
    ),
    "Johnny Juzang": (
        "https://www.spotrac.com/nba/player/_/id/78583/johnnyjuzang",
        "Spotrac reports a 2025-26 two-way contract value of $636,435.",
    ),
    "Chris Livingston": (
        "https://www.aiscore.com/salary/basketball/player-chris-livingston-jek3pse8nxu2qo2",
        "AiScore lists $2,296,274 with Milwaukee plus $527,611 with Cleveland; their sum is $2,823,885.",
    ),
    "Kevin McCullar Jr.": (
        "https://nbasalaries.com/players/kevin-mccullar-jr/",
        "NBA Salaries lists the 2025-26 two-way salary as $636,435.",
    ),
    "Antonio Reeves": (
        "https://nbaanalysis.net/charlotte-hornets-2025-26-payroll-revealed-lamelo-ball-must-earn-his-37-million-salary-with-strong-performances/",
        "NBAAnalysis payroll list reports $636,435 for Reeves.",
    ),
    "Orlando Robinson": (
        "https://www.hoopsrumors.com/2025/07/magic-sign-orlando-robinson-to-two-way-contract.html",
        "Hoops Rumors reports 2025-26 salary of $636,435.",
    ),
    "David Roddy": (
        "https://www.aol.com/articles/2026-nba-free-agent-rankings-143040000.html",
        "AOL/USA Today salary-history listing reports 2025-26 earnings of $128,686.",
    ),
    "Drew Peterson": (
        "https://m.aiscore.com/basketball/player-drew-peterson/oj7x6srg61ig7g3",
        "AiScore reports 2025-26 salary of $209,114.",
    ),
    "Daeqwon Plowden": (
        "https://m.aiscore.com/salary/basketball/player-daeqwon-plowden-o17p8sryp6fjkjw",
        "AiScore reports 2025-26 salary of $636,435.",
    ),
    "Tristen Newton": (
        "https://sports.yahoo.com/articles/uconns-tristen-newton-waived-houston-210927982.html",
        "Yahoo/CT Insider reports Newton earned $321,714 on his 2025-26 two-way deal.",
    ),
    "Drew Timme": (
        "https://www.aol.com/articles/2026-nba-free-agent-rankings-143040000.html",
        "AOL/USA Today reports 2025-26 earnings of $447,182; AiScore gives the same figure.",
    ),
    "Luke Travers": (
        "https://www.espn.com/nba/story/_/id/45445233/nba-offseason-2025-key-dates-deadlines-contracts",
        "ESPN’s 2025 offseason salary list reports $636,435.",
    ),
    "Stanley Umude": (
        "https://m.aiscore.com/salary/basketball/player-stanley-umude-g676jsvmdga9kry",
        "AiScore and EuroBasket report 2025-26 salary of $363,537.",
    ),
}

def recordOnlineSalaryChecks():
    pairs = pd.read_csv(PAIRS)
    existing = pd.read_csv(OVERRIDES)
    rows = []

    for player, (url, note) in ONLINE_SALARY_SOURCES.items():
        match = pairs[(pairs["player"].eq(player)) & pairs["target_season"].eq(2026)]
        if len(match) != 1:
            raise ValueError(f"Expected one salary row for {player} 2025-26; found {len(match)}.")
        row = match.iloc[0]
        rows.append({
            "player": player,
            "player_id": row["player_id"],
            "target_season": 2026,
            "target_salary": row["target_salary"],
            "verification_source_url": url,
            "verification_note": note,
        })

    ledger = pd.DataFrame(rows)
    ledger.to_csv(LEDGER, index=False)
    overrides = ledger[["player_id", "target_season", "target_salary", "verification_source_url"]].rename(
        columns={"verification_source_url": "salary_source_url"}
    )
    overrides["salary_needs_source_review"] = False
    overrides["salary_method"] = "verified_online_source_check"
    combined = pd.concat([existing, overrides], ignore_index=True)
    combined = combined.drop_duplicates(["player_id", "target_season"], keep="last")
    combined.to_csv(OVERRIDES, index=False)
    print(f"Recorded {len(ledger)} online salary checks.")
    print(f"Total persisted verified overrides: {len(combined)}.")

In [ ]:
runOnlineSalaryCheckWrite = False
if runOnlineSalaryCheckWrite: recordOnlineSalaryChecks()
print('Online salary override write enabled: ' + str(runOnlineSalaryCheckWrite))

## Safety notes

- Downloads and source checks require network access and are disabled by default.
- Salary audit and verification cells can write files under `data/raw/` and `data/processed/`; inspect the configured paths before enabling them.
- The PCA feature audit reads training rows only; it does not access development or test targets.